In [1]:
import requests
import time
import random
import hashlib
import smtplib
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- CONFIGURATION ---

URL_TO_CHECK = "https://www.marathondumedoc.com/inscription/"

# --- LISTE DES DESTINATAIRES ---

# Liste des personnes qui veulent recevoir le mode "PANIQUE" (Spammé toutes les 2 min)
VIP_EMAILS = ["petillion99@gmail.com"]

# Liste des personnes qui veulent juste être prévenues (1 seul mail)
STANDARD_EMAILS = ["quentinlevdso@gmail.com"]

# Configuration Email (Gmail - Expéditeur)
EMAIL_SENDER = "pierre.elipse@gmail.com"
EMAIL_PASSWORD = "mifz iexy csrq xsjr"

# Intervalles de surveillance normaux
MIN_INTERVAL = 50
MAX_INTERVAL = 70

# --- FONCTIONS ---

def get_content_hash(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            content = response.text
            return hashlib.md5(content.encode('utf-8')).hexdigest()
        return None
    except Exception as e:
        print(f"Erreur scrapping : {e}")
        return None

def send_email(recipients_list, subject, message):
    """
    Fonction générique pour envoyer un email à une liste de destinataires.
    Envoie le mail une seule fois.
    """
    if not recipients_list:
        return

    print(f"Envoi email à {len(recipients_list)} personne(s) : {subject}")

    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)

        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        # On joint la liste pour le champ "To"
        msg['To'] = ", ".join(recipients_list)
        msg['Subject'] = subject
        msg.attach(MIMEText(message, 'plain'))

        server.send_message(msg)
        server.quit()
        print(f"-> Email envoyé avec succès.")
    except Exception as e:
        print(f"Erreur Email : {e}")

def send_urgent_bombardment(subject, message):
    """
    Envoie l'email en mode BOMBARDEMENT (toutes les 2 min)
    SEULEMENT à la liste VIP_EMAILS.
    """

    urgent_subject = "MARATHON DU MEDOC OUVERT PREND TA PLACE"
    urgent_body = f"!!! URGENT !!! URGENT !!! URGENT !!!\n\n{message.upper()}\n\n\nLIEN DIRECT : {URL_TO_CHECK}\n\nALLEZ-Y MAINTENANT !!!"

    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    max_duration_minutes = 10
    start_time = time.time()
    count = 1

    print("\n" + "#"*50)
    print(f"!!! MODE BOMBARDEMENT POUR {VIP_EMAILS} !!!")
    print("#"*50 + "\n")

    while (time.time() - start_time) < (max_duration_minutes * 60):
        try:
            server = smtplib.SMTP(smtp_server, smtp_port)
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)

            msg = MIMEMultipart()
            msg['From'] = EMAIL_SENDER
            msg['To'] = ", ".join(VIP_EMAILS) # Uniquement les VIPs ici
            msg['Subject'] = urgent_subject
            msg.attach(MIMEText(urgent_body, 'plain'))

            server.send_message(msg)
            server.quit()
            print(f"[SPAM] Email VIOLENT n°{count} envoyé aux VIPs !")

        except Exception as e:
            print(f"Erreur envoi email : {e}")

        if (time.time() - start_time) < ((max_duration_minutes - 2) * 60):
            print("Attente 2 minutes avant prochain envoi...")
            time.sleep(120)
            count += 1

    print("Fin du cycle de bombardement.")

# --- PROGRAMME PRINCIPAL ---

def main():
    print(f"--- DÉMARRAGE DU SCRIPT ---")

    # 1. Email de démarrage pour TOUT LE MONDE (VIP + Standard)
    all_start = VIP_EMAILS + STANDARD_EMAILS
    send_email(all_start, "[MARATHON] Script Démarré", "Le script de surveillance est actif.")

    current_hash = get_content_hash(URL_TO_CHECK)
    if current_hash is None:
        print("Erreur critique : Impossible de récupérer la page initiale. Arrêt.")
        return

    print("Référence initiale stockée. Surveillance en cours...")
    last_heartbeat_date = None

    while True:
        now = datetime.now()
        current_date = now.date()

        # 2. Heartbeat 9h00 pour TOUT LE MONDE
        if now.hour >= 9 and last_heartbeat_date != current_date:
            send_email(all_start, "[MARATHON] Point de contrôle 9h00", "Le script tourne toujours.")
            last_heartbeat_date = current_date

        # Scrapping
        sleep_time = random.uniform(MIN_INTERVAL, MAX_INTERVAL)
        # On n'affiche plus l'heure à chaque boucle pour ne pas polluer, ou on la garde
        # time.sleep(sleep_time)
        print(f"[{now.strftime('%H:%M:%S')}] Check dans {int(sleep_time)} sec...")
        time.sleep(sleep_time)

        new_hash = get_content_hash(URL_TO_CHECK)

        if new_hash and new_hash != current_hash:
            alert_msg = "LA PAGE D'INSCRIPTION A CHANGÉ ! LES PLACES SONT PEUT-ÊTRE OUVERTES !"
            alert_subject = "MARATHON DU MEDOC OUVERT PREND TA PLACE"

            # STRATÉGIE D'ALERTE :

            # A) On prévient tout le monde IMMÉDIATEMENT (1 seule fois)
            # Cela inclus Pierre (VIP) et Quentin (Standard)
            send_email(VIP_EMAILS + STANDARD_EMAILS, alert_subject, alert_msg)

            # B) On lance le mode BOMBARDEMENT uniquement pour Pierre (VIP)
            # Cela va durer 10 minutes et spammer son téléphone
            send_urgent_bombardment(alert_subject, alert_msg)

            current_hash = new_hash
            print("Surveillance reprise...")
        elif new_hash == current_hash:
            pass

if __name__ == "__main__":
    main()

--- DÉMARRAGE DU SCRIPT ---
Envoi email à 2 personne(s) : [MARATHON] Script Démarré
-> Email envoyé avec succès.
Référence initiale stockée. Surveillance en cours...
Envoi email à 2 personne(s) : [MARATHON] Point de contrôle 9h00
-> Email envoyé avec succès.
[09:15:53] Check dans 50 sec...
[09:16:49] Check dans 52 sec...


KeyboardInterrupt: 